In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from brevitas.quant_tensor.int_quant_tensor import IntQuantTensor
from torch import Tensor
from utils import QuantModel, get_mnist_dataloaders, test_model, train_for_epoch

DATASET_PATH = "/home/a.redding/workspace/datasets"
EXPORT_PATH = "quant.pt"

In [ ]:
learning_rate = 0.01
momentum = 0.5
device = "cuda:0"
model = QuantModel(input_bit_width=4, weight_bit_width=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)

In [ ]:
train_loader, test_loader = get_mnist_dataloaders(DATASET_PATH)

In [ ]:
for epoch in range(10):
    train_for_epoch(model, device, train_loader, criterion, optimizer)
    test_model(model, device, test_loader, F.cross_entropy)

In [ ]:
model_weights: Tensor
model_scale: Tensor
correct_inputs: list[Tensor] = []
input_scale: Tensor = None
correct_targets: list[Tensor] = []

with torch.no_grad():
    for images, targets in test_loader:
        images, targets = images.to(device), targets.to(device)

        logits = model(images)
        preds = logits.argmax(dim=1, keepdim=True).squeeze()

        for i, correct in enumerate(preds == targets):
            if correct:  # Get quantized image
                x_quant: IntQuantTensor = model.fc1.input_quant(torch.flatten(images[i]))
                if input_scale is None:
                    input_scale = x_quant.scale.cpu()

                correct_inputs.append(x_quant.int().cpu())
                correct_targets.append(targets[i].cpu())

    # Get weights last for faster inference
    quant_weight = model.cpu().fc1.quant_weight()
    model_weights = quant_weight.int()
    model_scale = quant_weight.scale

correct_inputs = torch.vstack(correct_inputs)
correct_targets = torch.vstack(correct_targets)

In [ ]:
torch.save(
    {
        "model_weights": model_weights,
        "model_scale": model_scale,
        "correct_inputs": correct_inputs,
        "input_scale": input_scale,
        "correct_targets": correct_targets,
    },
    EXPORT_PATH,
)